Neural Networks From Scratch

In [1]:
import torch
import torch.nn as nn

def dot(v, w):
    return sum(vi * wi for vi, wi in zip(v, w))

#we start with a perceptron

#step function of a perceptron
def step_function(x):
    return 1 if x >= 0 else 0

#perceptron output calculation
def perceptron_output(weights, bias, x):
    #returns 1 if the perceptron fires and 0 if it does not
    calculation = (weights @ x) + bias
    return step_function(calculation)

In [2]:
weights = torch.tensor([2, 2])
bias = -3
x = torch.tensor([0, 0])

perceptron_output(weights, bias, x)

0

We use the sigmoid function instead of the step function above because of the need to use calculus and since the sigmoid function is a smooth approximation of the step function it serves our purpose. 

In [3]:
import math
#define the sigmoid which is a smooth approximation of the step function
def sigmoid(t):
    return 1 / (1 + math.exp(-t))

So now using the sigmoid function instead of the step function our neuron output looks like this.

In [4]:
def neuron_output(weights, inputs):
    # return sigmoid(weights @ inputs)
    return sigmoid(dot(weights, inputs))

In [5]:
neuron_output(torch.tensor([2, 2, -3]), torch.tensor([0, 0, 1]))

0.04742587317756678

We can represent a neural network as a list (layers) of lists (neurons) of lists (weights).

In [ ]:
#neurons are a list of weights and bias
neuron1 = [1, 0, 0, 1]
neuron2 = [1, 0, 1, 1]
neuron3 = [0, 0, 0, 0]
neuron4 = [0, 1, 0, 1]
neuron5 = [1, 1, 0, 0]
neuron6 = [0, 1, 1, 0]
neuron7 = [0, 1, 0, 0]
neuron8 = [1, 0, 0, 1]
neuron9 = [1, 1, 1, 1]
#layers are a list of neurons
layer1 = [
    neuron1,
    neuron2,
    neuron3,
    neuron4
]
layer2 = [
    neuron5,
    neuron6,
    neuron7,
    neuron8
]
output_layer = [
    neuron9
]
#a neural network is a list of layers
network = [
    layer1,
    layer2,
    output_layer
]
#network visualized is a list of lists of lists
network

In [6]:
#building a feed forward neural network using this logic
def feed_forward(neural_network, input_vector):
    """Takes in a neural network (list of lists of lists of weights) and returns the output."""
    #build our outputs list
    outputs = []
    #process one layer at a time
    for layer in neural_network:
        #build our input layer with bias
        input_with_bias = input_vector + [1]
        #run sigmoid with dot product on each neuron within a layer
        output = [neuron_output(neuron, input_with_bias) for neuron in layer]
        #we will append to the output
        outputs.append(output)
        #then the input to the next layer is the output of the current layer
        input_vector = output
    return outputs

In [7]:
#building an xor network
layer3 = [
    [20, 20, -30],
    [20, 20, -10]
]
layer4 = [
    [-60, 60, -30]
]
xor_network = [
    layer3,
    layer4
]

for x in [0, 1]:
    for y in [0, 1]:
        print(x, y, feed_forward(xor_network, [x, y])[-1])

0 0 [9.38314668300676e-14]
0 1 [0.9999999999999059]
1 0 [0.9999999999999059]
1 1 [9.383146683006828e-14]


Backpropagation is an algorithm used to train the weights within the neurons of the neural network based on target outputs. The algorithm goes like this:
1. Run the feed forward computation on the input vector in order to attain some initial output.
2. Compute the error of these outputs when compared to the target output.
3. Compute the gradient in order to adjust weights into direction that most decreases the error. 
4. Propogate these errors backwards in order to infer errors for the hidden layer.
5. Compute the gradients of these errors in order to adjust the weights of the neurons within the hidden layers.

In [ ]:
def backpropagate(network, input_vector, targets):
    #gather the outputs for all layers in the network
    hidden_outputs, outputs = feed_forward(network, input_vector)
    #use the derivative of the sigmoid to calculate output_deltas
    output_deltas = [output * (1 - output) * (output - target) for output, target in zip(outputs, targets)]
    #adjust the weights for the output layer on neuron at a time
    for i, output_neuron in enumerate(network[-1]):
        #focus on the ith output layer neuron
        for j, hidden_output in enumerate(hidden_outputs + [1]):
            #adjust the jth weight ased on both this neurons delta and its jth input
            output_neuron[j] -= output_deltas[i] * hidden_output
    #backpropagate errors to hidden layer
    hidden_deltas = [hidden_output * (1 - hidden_output) * dot(output_deltas, [n[i] for n in network[-1]])
                     for i, hidden_output in enumerate(hidden_outputs)]
    #adjust the weights for the hidden layer
    for i, hidden_neuron in enumerate(network[0]):
        for j, input in enumerate(input_vector + [1]):
            hidden_neuron[i] -= hidden_deltas[i] * input

In [28]:
for i, hidden_neuron in enumerate(xor_network[0]):
    print(1, hidden_neuron)

1 [20, 20, -30]
1 [20, 20, -10]


In [20]:
hidden_outputs, outputs = feed_forward(xor_network, [1, 1])

In [11]:
def calculate_delta(output, target):
    return output * (1 - output) * (output - target)

output_deltas = [calculate_delta(outputs[0], 1)]

In [23]:
hidden_deltas = []

for i, hidden_output in enumerate(hidden_outputs):
    calculate_dot = dot(output_deltas, [n[i] for n in xor_network[-1]])
    hidden_deltas.append(hidden_output * (1 - hidden_output) * calculate_dot)

In [27]:
for n in xor_network[-1]:
    print(n)

[-60, 60, -30]


In [24]:
hidden_deltas

[2.5557331366771064e-16, -5.262863150005922e-25]

In [26]:
0.99999 * (1 - 0.99999) * dot([-9.383e-14], [60.000001])

-5.629743795803441e-17

In [ ]:
hidden_deltas = [hidden_output * (1 - hidden_output) * dot(output_deltas, [n[i] for n in network[-1]])
                     for i, hidden_output in enumerate(hidden_outputs)]